In [6]:
# I learned the approach implemented in this notebook
# https://www.kaggle.com/code/yuriygreben/birdclef-26-onnx-perch-dual-ssms-vectorized-mlp
# will try to implemented some of the ideas of the notebook.

In [ ]:
%pip install -q --no-deps /kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

In [11]:
mode = 'train_offline'

In [16]:
import librosa
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from pathlib import Path
if "offline" in mode:
    print('working offline')
    import tensorflow as tf
    import math
    import soundfile as sf
    import mlflow
    tf.config.set_visible_devices([], 'GPU')  # force CPU
elif "online" in mode or mode == 'submit':
    %pip install -q --no-deps /kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
import onnxruntime as ort

working offline


## Download perch_v2_cpu and run it and cache the results

First, try to run ssm on the audio file in train_soundscapes folder

In [44]:
kaggle_onnx_path = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/perch_v2.onnx')
# dictionary to perch_v2 for different modes
onnx_path_dict = {
    'train_offline': Path('../models/perch_onnx/perch_v2.onnx'),
    'train_online': kaggle_onnx_path,
    'submit': kaggle_onnx_path,
    'test_submit': kaggle_onnx_path,
}
kaggle_base_path = Path('/kaggle/input/competitions/birdclef-2026')
base_path_dict = {
    'train_offline': Path('../data'),
    'train_online': kaggle_base_path,
    'submit': kaggle_base_path,
    'test_submit': kaggle_base_path,
}
onnx_path = onnx_path_dict[mode]
base_path = base_path_dict[mode]

In [45]:
session_option = ort.SessionOptions()
session_option.intra_op_num_threads = 4
onnx_session = ort.InferenceSession(onnx_path, session_options = session_option, providers = ['CPUExecutionProvider'])
onnx_ipt_name = onnx_session.get_inputs()[0].name
print(f'Onnx input ame: {onnx_ipt_name}')
onnx_opt_map = {o.name: i for i, o in enumerate(onnx_session.get_outputs())}
print(f'Onnx output maps: {onnx_opt_map}')

Onnx input ame: inputs
Onnx output maps: {'embedding': 0, 'spatial_embedding': 1, 'spectrogram': 2, 'label': 3}


### Build soundscapes cache

In [46]:
SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: float) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    Arguments:
        path: path to .ogg file
        offset_sec: offset in seconds into the file
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
sample_path = base_path / 'train_soundscapes' / 'BC2026_Train_0001_S08_20250606_030007.ogg'
chunk = load_chunk(sample_path, offset_sec=0.0)
print('Shape:', chunk.shape)   # (160000,)
print('dtype:', chunk.dtype)   # float32
print('Range:', chunk.min(), chunk.max())

Shape: (160000,)
dtype: float32
Range: -0.19364518 0.19562061


In [52]:
def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = waveform[np.newaxis, :]
    outs = onnx_session.run(None, {onnx_ipt_name: inp})
    print(len(outs))
    emb  = outs[onnx_opt_map['embedding']].astype(np.float32)
    label_logits = outs[onnx_opt_map['label']]  # (1, 508)

    
    return emb[0], label_logits[0]  # (1536,)

embedding, label_logtis = extract_embedding(chunk)
print(f'Embedding shape: {embedding.shape}')
print(f'Label logits shape: {label_logtis.shape}')

4
Embedding shape: (1536,)
Label logits shape: (14795,)


### Get the sites and the hours where and when sounds are recorded
Sites and the hours will be added as additional features along with the features obtained from perch-v2

In [53]:
import re
# train_soundscapes file names has site up to 20, so this pattern is used
# if it does not match, it means the files in test_sounscapes folder may have different patterns
filename_pattern = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(\w{3})_(\w{8})_(\w{6})\.ogg")
def get_site_month_hour(filename) -> tuple[str, int,int]:
    """
    Arg:
    filename: the filename of a file in soundscape folder that has site and hour information
    Return:
    A tuple with filename as string and site as an integer
    """
    _, site, date, hour = filename_pattern.match(filename).groups()

    return (site, int(date[4:6]), int(hour[:2]))

    
def build_file_name_arr(soundscapes_folder_path):
    return 0
    
get_site_month_hour('BC2026_Train_0001_S08_20250606_030007.ogg')


('S08', 6, 3)

In [61]:

train_sounscapes_label_df = pd.read_csv(base_path / 'train_soundscapes_labels.csv')
train_sounscapes_label_df['site'], train_sounscapes_label_df['month'], train_sounscapes_label_df['hour'] = zip(*train_sounscapes_label_df['filename'].apply(get_site_month_hour))

sites = sorted(train_sounscapes_label_df['site'].unique())
hours = sorted(train_sounscapes_label_df['hour'].unique())
months = sorted(train_sounscapes_label_df['month'].unique())
print(f'sites: {sites} \n hours: {hours}\n months: {months}')

sites: ['S03', 'S08', 'S09', 'S13', 'S15', 'S18', 'S19', 'S22', 'S23'] 
 hours: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
 months: [np.int64(1), np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(10), np.int64(11), np.int64(12)]


In [54]:

if "offline" in mode:
    perch_label_path = Path('../models/perch_onnx/labels.csv')
    
elif "online" in mode or mode == 'submit':
    perch_label_path = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/labels.csv')

perch_df = pd.read_csv(perch_label_path)
# rename the column to match the column name in taxonomy.csv
perch_df.rename(columns={'inat2024_fsd50k': 'scientific_name'}, inplace=True)
perch_df.head()

,scientific_name
0,Abavorana luctuosa
1,Abeillia abeillei
2,Abroscopus albogularis
3,Abroscopus schisticeps
4,Abroscopus superciliaris


In [55]:
taxonomy_df = pd.read_csv(base_path / 'taxonomy.csv')
taxonomy_df.head()

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia


In [56]:
taxonomy_join_perch_df = taxonomy_df.merge(perch_df.rename_axis("perch_idx").reset_index(), on='scientific_name', how='left')
taxonomy_join_perch_df.head()


,primary_label,inat_taxon_id,scientific_name,common_name,class_name,perch_idx
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta,5743.0
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia,NaN
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia,7018.0
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia,NaN
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia,NaN


In [42]:
nan_mask = taxonomy_join_perch_df['perch_idx'].isna().sum()
print(f"Number of nan in perch_idx: {nan_mask}")

Number of nan in perch_idx: 31


In [43]:
# fill NaN in 'perch_idx' with len(perch_df)-an unknown species
unknow_species_idx = len(perch_df)
taxonomy_join_perch_df['perch_idx'] = taxonomy_join_perch_df['perch_idx'].fillna(unknow_species_idx)
taxonomy_join_perch_df['perch_idx'] = taxonomy_join_perch_df['perch_idx'].astype(int)
taxonomy_join_perch_df.head()

,primary_label,inat_taxon_id,scientific_name,common_name,class_name,perch_idx
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta,5743
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia,14795
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia,7018
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia,14795
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia,14795


### Build model